# Ylivertainen v2 — Clinical Association Pipeline

End-to-end workflow for finding statistically valid associations between **target outcomes**
and **predictor variables** in a clinical dataset.

This notebook drives six universal modules:

| Module                       | Purpose                                                    |
|------------------------------|------------------------------------------------------------|
| `schema_infer.py`            | Auto-classify each column (continuous, ordinal, …)         |
| `cleaning.py`                | Apply the schema, audit duplicates, derive new columns     |
| `dda.py`                     | Per-column descriptive stats + SVG plots                   |
| `missingness_resolution.py`  | Missing pattern analysis, flags, MICE multiple imputation  |
| `eda.py`                     | Univariate target × predictor screening (FDR-corrected)    |
| `inferential.py`             | Multivariable logistic regression with Rubin pooling       |

**Pipeline order**

```
load → infer schema → clean → DDA → missingness → derive new cols → DDA again
   → EDA screen → MICE impute → multivariable logistic (Rubin pool) → outputs
```

All outputs land under `output/<stage>/{figures,tables}/` as SVG and CSV.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from schema_infer import infer_schema, print_schema_template, schema_summary, ColSpec
from cleaning import (apply_schema, audit_duplicates,
                      bin_numeric, bin_datetime, make_missing_flag, combine_categories)
from dda import run_dda
from missingness_resolution import (analyze_missingness, add_missing_flags,
                                    mark_structural_missing, drop_rows,
                                    mice_impute, simple_impute)
from eda import screen_associations
from inferential import run_inferential

OUTPUT_ROOT = Path("output")


## 1. Load your data

Change the path to point at your Excel/CSV file. The rest of the notebook is dataset-agnostic.


In [2]:
DATA_PATH = "RPE 2020-2025.xlsx"   # or "yourdata.csv"

if str(DATA_PATH).endswith(".csv"):
    df_raw = pd.read_csv(DATA_PATH)
else:
    df_raw = pd.read_excel(DATA_PATH)

print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head()


Loaded: 1168 rows × 20 columns


,gads,Personas kods,Vecums,PSA līmenis pirms biops,MRI lesions 1 PIRADS,MRI lesions 2 PIRADS,MRI lesions 3 PIRADS,TNM pirms op. (MDK slēdziens),"Riska grupa (zema -1; vidēja-2, augsta-3 )",gleason grade biopsijā,biospsijas veids,RPE grade,Laiks no biopsijas (veikšanas) līdz operācijai,Pvol,upgrade,upstage,downgrade,RPE TNM,PSA bīlvums,R1?
0,2020,280362-12350,57,16.20,5,0,0,T2cN0M0,augsta,2,transrektāla,2,194,41.0,0,0,0,T2cN0M0,0.395122,0
1,2020,010855-11322,64,10.40,0,0,0,T2N1M0,augsta,2,transrektāla,2,169,26.2,0,0,0,T2cN0M0,0.396947,0
2,2020,100474-10701,45,5.55,3,2,0,T2aN0M0,zema,1,transrektāla,1,142,38.0,0,0,0,T2aN0M0,0.146053,0
3,2020,061143-11286,76,7.24,5,3,3,T2cN0M0,augsta,2,transrektāla,2,107,60.0,0,0,0,T2cN0M0,0.120667,0
4,2020,030658-11140,61,7.20,5,0,0,T3aN0M0,augsta,1,transrektāla,1,213,50.7,0,0,0,T2cN0M0,0.142012,0


### 1a. (Optional) Rename columns to clean snake_case

If your source has long Latvian/Russian/free-text column names, rename them here.
Comment this cell out for new datasets where column names are already clean.


In [3]:
# Example for RPE study — edit/remove for other datasets:
# df_raw.columns = ['year', 'pk', 'age', 'preop_PSA', 'lesion_1_MRI_PIRADS', ...]
# df_raw.columns = [c.strip() for c in df_raw.columns]

df_raw.columns = ['year', 'pk', 'age', 'preop_PSA',
       'lesion_1_MRI_PIRADS', 'lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS',
       'preop_TNM_MDK',
       'risk_group', 'biopsy_gleason_grade',
       'biopsy_type', 'RPE_grade',
       'biopsy_to_RPE_days', 'prostate_volume', 'upgrade',
       'upstage', 'downgrade', 'RPE_TNM', 'PSA_density', 'resection_lines_pos']

df = df_raw.reindex([
    'year', 'biopsy_type',
    'lesion_1_MRI_PIRADS', 'lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS',
    'age', 'preop_PSA', 'preop_TNM_MDK',
    'risk_group', 'biopsy_gleason_grade',
    'RPE_grade',
    'biopsy_to_RPE_days', 'prostate_volume', 'upgrade',
    'upstage', 'downgrade', 'RPE_TNM', 'PSA_density', 'resection_lines_pos', 'pk'], axis=1)

df.head(2)

,year,biopsy_type,lesion_1_MRI_PIRADS,lesion_2_MRI_PIRADS,lesion_3_MRI_PIRADS,age,preop_PSA,preop_TNM_MDK,risk_group,biopsy_gleason_grade,RPE_grade,biopsy_to_RPE_days,prostate_volume,upgrade,upstage,downgrade,RPE_TNM,PSA_density,resection_lines_pos,pk
0,2020,transrektāla,5,0,0,57,16.2,T2cN0M0,augsta,2,2,194,41.0,0,0,0,T2cN0M0,0.395122,0,280362-12350
1,2020,transrektāla,0,0,0,64,10.4,T2N1M0,augsta,2,2,169,26.2,0,0,0,T2cN0M0,0.396947,0,010855-11322


## 2. Infer schema and override

The engine auto-classifies every column. Inspect the printed dict, copy it into the
next cell, edit anything wrong (e.g. force `lesion_2_MRI_PIRADS` to `ordinal`,
mark `pk` as `id`, drop a junk column with `kind="skip"`).


In [4]:
schema = infer_schema(df_raw)
schema_summary(schema)


,column,kind,keep,ordered_levels,nulls,note
0,year,ordinal,True,"[2020, 2021, 2022, 2023, 2024, 2025]",None,
1,pk,id,True,None,None,
2,age,continuous,True,None,None,
3,preop_PSA,continuous,True,None,None,
4,lesion_1_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
5,lesion_2_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
6,lesion_3_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
7,preop_TNM_MDK,text,True,None,None,
8,risk_group,nominal,True,None,None,
9,biopsy_gleason_grade,ordinal,True,"[1, 2, 3, 4, 5]",None,


In [5]:
# Print a paste-back-able template; edit it in the next cell.
print_schema_template(schema);

schema_overrides = {
    'year': ColSpec(name='year', kind='ordinal', ordered_levels=[2020, 2021, 2022, 2023, 2024, 2025]),
    'pk': ColSpec(name='pk', kind='id'),
    'age': ColSpec(name='age', kind='continuous'),
    'preop_PSA': ColSpec(name='preop_PSA', kind='continuous'),
    'lesion_1_MRI_PIRADS': ColSpec(name='lesion_1_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'lesion_2_MRI_PIRADS': ColSpec(name='lesion_2_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'lesion_3_MRI_PIRADS': ColSpec(name='lesion_3_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'preop_TNM_MDK': ColSpec(name='preop_TNM_MDK', kind='text'),
    'risk_group': ColSpec(name='risk_group', kind='nominal'),
    'biopsy_gleason_grade': ColSpec(name='biopsy_gleason_grade', kind='ordinal', ordered_levels=[1, 2, 3, 4, 5]),
    'biopsy_type': ColSpec(name='biopsy_type', kind='nominal'),
    'RPE_grade': ColSpec(name='RPE_grade', kind='ordinal', ordered_levels=[1, 2,

### 2a. Paste the edited schema below

Take the printout from the cell above, paste it here, and adjust kinds/ordered_levels
as needed. Anything you don't override stays as inferred.

For RPE specifically, things to check:
- `pk` → `id`
- `preop_TNM_MDK`, `RPE_TNM` → `nominal`
- `risk_group` → `ordinal` with `ordered_levels=['zema','vidēja','augsta']`
- `biopsy_gleason_grade`, `RPE_grade`, PIRADS columns → `ordinal` with `[1,2,3,4,5]` (or `[0,1,2,3,4,5]`)
- `upgrade`, `upstage`, `downgrade`, `resection_lines_pos` → `binary`


In [6]:
# Example override — adapt to your dataset!
schema_overrides = {
    'year': ColSpec(name='year', kind='ordinal', ordered_levels=[2020, 2021, 2022, 2023, 2024, 2025]),
    'pk': ColSpec(name='pk', kind='id'),
    'age': ColSpec(name='age', kind='continuous', note="create bins"),
    'preop_PSA': ColSpec(name='preop_PSA', kind='continuous'),
    'lesion_1_MRI_PIRADS': ColSpec(name='lesion_1_MRI_PIRADS', kind='ordinal', ordered_levels=[2, 3, 4, 5], nulls=(0,), note="delete NaN rows"),
    'lesion_2_MRI_PIRADS': ColSpec(name='lesion_2_MRI_PIRADS', kind='ordinal', ordered_levels=[2, 3, 4, 5], nulls=(0,), note="NaN rows - no 2nd lesion"),
    'lesion_3_MRI_PIRADS': ColSpec(name='lesion_3_MRI_PIRADS', kind='ordinal', ordered_levels=[2, 3, 4, 5], nulls=(0,), note="NaN rows - no 3rd lesion"),
    'preop_TNM_MDK': ColSpec(name='preop_TNM_MDK', kind='nominal'),
    'risk_group': ColSpec(name='risk_group', kind='ordinal', ordered_levels=['zema', 'vidēja', 'augsta']),
    'biopsy_gleason_grade': ColSpec(name='biopsy_gleason_grade', kind='ordinal', ordered_levels=[1, 2, 3, 4, 5]),
    'biopsy_type': ColSpec(name='biopsy_type', kind='nominal'),
    'RPE_grade': ColSpec(name='RPE_grade', kind='ordinal', ordered_levels=[1, 2, 3, 4, 5]),
    'biopsy_to_RPE_days': ColSpec(name='biopsy_to_RPE_days', kind='continuous', note="create bins"),
    'prostate_volume': ColSpec(name='prostate_volume', kind='continuous'),
    'upgrade': ColSpec(name='upgrade', kind='binary'),
    'upstage': ColSpec(name='upstage', kind='binary'),
    'downgrade': ColSpec(name='downgrade', kind='binary'),
    'RPE_TNM': ColSpec(name='RPE_TNM', kind='nominal'),
    'PSA_density': ColSpec(name='PSA_density', kind='continuous'),
    'resection_lines_pos': ColSpec(name='resection_lines_pos', kind='binary'),
}
# merge overrides on top of inferred schema:
schema.update(schema_overrides)
schema_summary(schema)


,column,kind,keep,ordered_levels,nulls,note
0,year,ordinal,True,"[2020, 2021, 2022, 2023, 2024, 2025]",None,
1,pk,id,True,None,None,
2,age,continuous,True,None,None,create bins
3,preop_PSA,continuous,True,None,None,
4,lesion_1_MRI_PIRADS,ordinal,True,"[2, 3, 4, 5]",[0],delete NaN rows
5,lesion_2_MRI_PIRADS,ordinal,True,"[2, 3, 4, 5]",[0],NaN rows - no 2nd lesion
6,lesion_3_MRI_PIRADS,ordinal,True,"[2, 3, 4, 5]",[0],NaN rows - no 3rd lesion
7,preop_TNM_MDK,nominal,True,None,None,
8,risk_group,ordinal,True,"[zema, vidēja, augsta]",None,
9,biopsy_gleason_grade,ordinal,True,"[1, 2, 3, 4, 5]",None,


## 3. Apply schema → coerce dtypes, replacements, nulls

This is the only place dtypes are set. Downstream stages trust the schema.


In [7]:
df = apply_schema(df_raw, schema)
df.dtypes

year                    category
pk                        string
age                        int64
preop_PSA                float64
lesion_1_MRI_PIRADS     category
lesion_2_MRI_PIRADS     category
lesion_3_MRI_PIRADS     category
preop_TNM_MDK           category
risk_group              category
biopsy_gleason_grade    category
biopsy_type             category
RPE_grade               category
biopsy_to_RPE_days         int64
prostate_volume          float64
upgrade                  boolean
upstage                  boolean
downgrade                boolean
RPE_TNM                 category
PSA_density              float64
resection_lines_pos      boolean
dtype: object

## 4. Duplicate audit

Provide ID columns. The audit returns rows in duplicate groups and a cleaned frame.


In [8]:
ID_COLS = ['pk', 'year']   # edit for your dataset

dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
print(f"Found {len(dupes)} duplicated rows (across {len(dupes)//2 if len(dupes) else 0}+ groups)")
dupes.head()


Found 0 duplicated rows (across 0+ groups)


,year,pk,age,preop_PSA,lesion_1_MRI_PIRADS,lesion_2_MRI_PIRADS,lesion_3_MRI_PIRADS,preop_TNM_MDK,risk_group,biopsy_gleason_grade,biopsy_type,RPE_grade,biopsy_to_RPE_days,prostate_volume,upgrade,upstage,downgrade,RPE_TNM,PSA_density,resection_lines_pos


## 4b. Row removal — drop invalid records

After dtypes are coerced and duplicates audited, this is the place to permanently
remove rows that should not be in analysis at all (data-entry errors, ineligible
patients, impossible values). Every drop is **logged** so the methods section of
your paper can quote exact counts.

Common reasons:
- Out-of-range values (e.g. `age < 18`, `preop_PSA < 0`, `biopsy_to_RPE_days < 0`)
- Wrong cohort (e.g. patients without a primary RPE)
- Records missing critical identifiers
- Failed sanity checks against source records

Use `where=` for readable pandas-query strings, or `mask=` for arbitrary boolean
Series. Add as many calls as you need.


In [9]:
# ─────────────────────────────────────────────────────────────────────────
# Row removal — delete rows that should not be in the analysis at all.
#
# This is for rows that are STRUCTURALLY WRONG (out-of-cohort patient, data-
# entry errors, impossible values), NOT for rows with missing values
# (missingness is handled in section 6). Every call appends an entry to
# drop_log so you can report exact counts in your paper's methods section.
#
# Two ways to specify which rows to drop:
#   where='age < 18'                    → pandas query string (most readable)
#   mask=df['RPE_grade'].isna() & ...   → arbitrary boolean Series (most powerful)
#
# Always provide a meaningful `reason` — it shows up in the audit log.
# ─────────────────────────────────────────────────────────────────────────

drop_log = []

# --- Examples below — edit / delete / uncomment for your dataset ---

df = drop_rows(df, mask=df['lesion_1_MRI_PIRADS'].isna(),
                reason='incorrect data, no lesion recorded', log=drop_log)

# df = drop_rows(df, where='preop_PSA < 0',
#                reason='negative PSA = data entry error', log=drop_log)

# df = drop_rows(df, where='biopsy_to_RPE_days < 0',
#                reason='surgery before biopsy = entry error', log=drop_log)

# df = drop_rows(df, mask=df['RPE_grade'].isna() & df['upgrade'].isna(),
#                reason='no histopathology recorded', log=drop_log)

# --- summary table ---
pd.DataFrame(drop_log) if drop_log else print('No rows dropped (uncomment examples above as needed)')


,reason,criterion,n_dropped,n_remaining
0,"incorrect data, no lesion recorded",mask,33,1135


## 5. DDA — first pass

Descriptive stats + SVG plots for every kept column.
Outputs land in `output/dda/`.


In [10]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT, skip_cols=[])

print("\n--- continuous ---");  display(dda_tables['continuous'])
print("\n--- categorical ---"); display(dda_tables['categorical'])
print("\n--- binary ---");      display(dda_tables['binary'])
print("\n--- datetime ---");    display(dda_tables['datetime'])



--- continuous ---


,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,age,continuous,1135,38,0.0,45.000000,53.000000,65.00000,64.721586,64.940594,74.000000,84.000000,67.00,6.526437,0.100839,9.000000,-0.316196,-0.138151
1,preop_PSA,continuous,1135,655,0.0,0.313000,4.100000,8.00000,10.863528,9.122073,26.660000,100.000000,8.00,8.985280,0.827105,6.400000,3.525050,18.546493
2,biopsy_to_RPE_days,continuous,1135,304,0.0,0.000000,62.000000,118.00000,171.007930,128.972497,426.300000,3326.000000,NaN,217.344469,1.270961,74.000000,6.718823,64.228463
3,prostate_volume,continuous,1135,320,0.0,2.000000,20.000000,39.00000,44.147031,41.316326,85.000000,178.000000,40.00,21.071803,0.477310,22.000000,1.690357,4.380664
4,PSA_density,continuous,1135,1089,0.0,0.020579,0.076774,0.20625,0.293196,0.237345,0.800784,4.545455,0.25,0.308676,1.052800,0.194922,5.815187,57.724241



--- categorical ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,max_class_imbalance,median_category,balance,entropy_bin
0,year,ordinal,True,1135,6,0.00,2022,18.94,2023,18.41,2024,1.43,2022,0.9955,2.5733
1,lesion_1_MRI_PIRADS,ordinal,True,1135,4,0.00,5,46.26,4,43.88,2,21.00,4,0.7206,1.4412
2,lesion_2_MRI_PIRADS,ordinal,True,288,4,74.63,4,60.42,3,25.00,2,13.38,4,0.7372,1.4745
3,lesion_3_MRI_PIRADS,ordinal,True,35,4,96.92,3,45.71,4,37.14,5,8.00,3,0.8203,1.6405
4,preop_TNM_MDK,nominal,False,1135,15,0.00,T2cN0M0,34.10,T2N0M0,23.44,T2N1M0,NaN,NaN,0.6922,2.7042
5,risk_group,ordinal,True,1135,3,0.00,vidēja,45.81,augsta,35.51,zema,2.45,vidēja,0.9454,1.4985
6,biopsy_gleason_grade,ordinal,True,1135,5,0.00,2,43.08,1,36.83,5,14.82,2,0.7739,1.7969
7,biopsy_type,nominal,False,1135,2,0.00,transrektāla,63.00,transperineāla,37.00,transperineāla,1.70,NaN,0.9507,0.9507
8,RPE_grade,ordinal,True,1135,5,0.00,2,60.18,3,15.77,4,16.66,2,0.7185,1.6683
9,RPE_TNM,nominal,False,1135,25,0.00,T3aN0M0,31.54,T2cN0M0,23.96,T4N0M0,358.00,NaN,0.5793,2.6901



--- binary ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,max_class_imbalance,median_category,balance,entropy_bin
0,upgrade,binary,False,1135,2,0.0,False,64.49,True,35.51,True,1.82,NaN,0.9385,0.9385
1,upstage,binary,False,1135,2,0.0,False,55.24,True,44.76,True,1.23,NaN,0.9921,0.9921
2,downgrade,binary,False,1135,2,0.0,False,88.02,True,11.98,True,7.35,NaN,0.5289,0.5289
3,resection_lines_pos,binary,False,1135,2,0.0,False,93.30,True,6.70,True,13.93,NaN,0.3545,0.3545



--- datetime ---


""


## 6. Missingness analysis

Per-column %, plus a Jaccard co-missingness heatmap so you can spot blocks of
columns that are missing together (often a data-entry-process artifact).


In [11]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary

,column,n_missing,pct_missing
0,lesion_3_MRI_PIRADS,1100,96.92
1,lesion_2_MRI_PIRADS,847,74.63
2,year,0,0.00
3,RPE_grade,0,0.00
4,PSA_density,0,0.00
5,RPE_TNM,0,0.00
6,downgrade,0,0.00
7,upstage,0,0.00
8,upgrade,0,0.00
9,prostate_volume,0,0.00


### 6b. Resolve missingness — tag structural vs MNAR

Not all `NaN` means the same thing. Before MICE imputes anything, classify each
column with missing values into one of three buckets:

| Bucket | Meaning | Action in this section |
|---|---|---|
| **Structural** | The value *does not exist* (e.g. `lesion_2_MRI_PIRADS` is NaN because the MRI showed only one lesion) | `mark_structural_missing(...)` — derives count + max features, flips originals to `kind='skip'` so MICE never touches them |
| **MNAR** | Missingness depends on the unobserved value itself (e.g. PSA not measured because the clinician judged it unnecessary) | `add_missing_flags(...)` in section 6c — adds an explicit `<col>_missing` flag, then imputes normally |
| **MAR** | Missingness depends only on *observed* variables | No special handling — MICE handles it correctly out of the box |

**Rule of thumb.** Ask: *"If this patient were re-examined today with perfect technique,
would a value exist?"* If **no** → structural. If **yes** → MAR/MNAR.

The structural step **must come before** MICE, because MICE will happily fabricate
PIRADS scores for non-existent lesions otherwise.


In [12]:
# ─────────────────────────────────────────────────────────────────────────
# STRUCTURAL_GROUPS — configure one entry per "slot family" in your data.
#
# A "slot family" is a set of columns that hold OPTIONAL repeats of the same
# clinical thing (lesion 1 / 2 / 3, tumour 1 / 2, biopsy core 1 / 2 / 3 …).
# NaN in slot 2 or 3 doesn't mean "we forgot to measure" — it means "that slot
# doesn't exist for this patient". Imputing it would invent findings.
#
# Each entry takes these keys:
#
#   'cols'         : list of all slot columns in the family (INCLUDE slot 1 here,
#                    so the count feature reflects the true number of slots).
#
#   'derive_count' : True  → create <group_name> = number of non-null slots.
#                    Use for "how many lesions did this patient have?"
#
#   'derive_max'   : True  → create <group_name>_max = max value across slots.
#                    Clinically the "dominant" lesion's PIRADS. Only set True
#                    when the slot columns are ORDINAL or NUMERIC.
#
#   'count_levels' : optional ordering for the count feature, e.g. [0,1,2,3].
#                    Sets the ordinal categories so charts/tables are ordered.
#
#   'max_levels'   : optional ordering for the max feature, e.g. [1,2,3,4,5].
#
#   'skip_after'   : list of columns to mark kind='skip' AFTER deriving features.
#                    IMPORTANT: usually leave the PRIMARY slot (lesion_1) OUT of
#                    this list — its value is real, not structural, and it stays
#                    as a normal predictor. Only skip the optional repeats.
#
# Effect: skipped columns are excluded from MICE imputation, EDA screening,
# the multivariable model, and the DDA second pass. They stay in the dataframe
# (you can still inspect them) but no statistic touches them.
# ─────────────────────────────────────────────────────────────────────────

STRUCTURAL_GROUPS = {
    # --- Example for RPE: 3 MRI lesion slots ---
     'n_lesions_MRI': {
         'cols':         ['lesion_1_MRI_PIRADS',
                          'lesion_2_MRI_PIRADS',
                          'lesion_3_MRI_PIRADS'],
         'derive_count': True,
         'derive_max':   True,
         'count_levels': [1, 2, 3],
         'max_levels':   [1, 2, 3, 4, 5],
         'skip_after':   ['lesion_2_MRI_PIRADS',   # ← optional slots only
                          'lesion_3_MRI_PIRADS'],  # ← lesion_1 stays as predictor
     },
}

if STRUCTURAL_GROUPS:
    df = mark_structural_missing(df, schema, STRUCTURAL_GROUPS)
    new_cols = [g for g in STRUCTURAL_GROUPS] + [f'{g}_max' for g in STRUCTURAL_GROUPS]
    print('Derived structural features:', [c for c in new_cols if c in df.columns])
    print('Now marked kind=\'skip\' (excluded from MICE / EDA / inferential):',
          [c for c, sp in schema.items() if sp.kind == 'skip'])
else:
    print('No structural-missing groups configured.')
    print('If your dataset has NaN that means "this slot does not exist",')
    print('edit STRUCTURAL_GROUPS above before running MICE in section 11.')


Derived structural features: ['n_lesions_MRI', 'n_lesions_MRI_max']
Now marked kind='skip' (excluded from MICE / EDA / inferential): ['lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS']


In [13]:
# Sanity check — verify the new derived columns look right
derived = [g for g in STRUCTURAL_GROUPS] + [f'{g}_max' for g in STRUCTURAL_GROUPS]
derived = [c for c in derived if c in df.columns]
if derived:
    display(df[derived].describe(include='all'))
    for c in derived:
        print(c, df[c].value_counts(dropna=False).to_dict())


,n_lesions_MRI,n_lesions_MRI_max
count,1135.0,1135.000000
mean,1.284581,4.344493
std,0.51527,0.712884
min,1.0,2.000000
25%,1.0,4.000000
50%,1.0,4.000000
75%,2.0,5.000000
max,3.0,5.000000


n_lesions_MRI {np.int64(1): 847, np.int64(2): 253, np.int64(3): 35}
n_lesions_MRI_max {5.0: 527, 4.0: 496, 3.0: 88, 2.0: 24}


### 6c. Add MNAR missingness flags

For columns where the missingness itself carries information (true MNAR — e.g.
PSA not measured *because* risk looked low), add an explicit boolean flag so
the model can use 'was-it-measured' as a predictor. The schema is updated
automatically — no need to register the flag columns yourself.


In [14]:
# ─────────────────────────────────────────────────────────────────────────
# MNAR_COLS — list ONLY columns that meet ALL THREE criteria:
#
#   1. The value EXISTS in reality (it's not structurally absent — those went
#      into STRUCTURAL_GROUPS in section 6b).
#   2. The value is sometimes NOT recorded.
#   3. The reason it wasn't recorded is plausibly TIED TO THE VALUE ITSELF
#      (e.g. PSA not measured BECAUSE the clinician thought the patient was
#      low-risk → low-risk patients are systematically missing → MNAR).
#
# If missingness is purely due to data-entry chaos / random clerical loss,
# that's MAR — leave the column OUT of MNAR_COLS; MICE handles it correctly
# without a flag.
#
# Effect: for each column listed here, a new boolean column <col>_missing is
# added (True where the original was NaN) and registered in the schema as a
# binary predictor. The multivariable model can then use the FACT of missing-
# ness as its own predictor, separately from the imputed value.
# ─────────────────────────────────────────────────────────────────────────

MNAR_COLS = []   # e.g. ['preop_PSA', 'PSA_density']

df = add_missing_flags(df, MNAR_COLS, schema=schema)
df.filter(like='_missing').head()


""
0
2
3
4
5


## 7. Derive new columns (age bins, time bins, PSA categories…)

Use the helpers below freely. Any new column you add **must** also be added to the
schema so DDA/EDA/inferential will analyze it.


In [15]:
# ── 1. biopsy → RPE, in months ───────────────────────────────────────────
if 'biopsy_to_RPE_days' in df.columns:
    df['biopsy_to_RPE_month_bin'] = bin_numeric(
        df['biopsy_to_RPE_days'],
        bins=[-np.inf, 90, 180, 270, 365, np.inf],
        labels=['<3 mo', '3-6 mo', '6-9 mo', '9-12 mo', '>12 mo'],
    )
    schema['biopsy_to_RPE_month_bin'] = ColSpec(
        name='biopsy_to_RPE_month_bin', kind='ordinal',
        ordered_levels=['<3 mo', '3-6 mo', '6-9 mo', '9-12 mo', '>12 mo'],
    )

# ── 2. PSA density, clinical 0.15 cutoff ─────────────────────────────────
if 'PSA_density' in df.columns:
    df['PSAD_bin'] = bin_numeric(
        df['PSA_density'],
        bins=[-np.inf, 0.15, np.inf],
        labels=['<0.15', '≥0.15'],
    )
    schema['PSAD_bin'] = ColSpec(
        name='PSAD_bin', kind='ordinal',
        ordered_levels=['<0.15', '≥0.15'],
    )

# ── 3. Prostate volume (ml) ──────────────────────────────────────────────
if 'prostate_volume' in df.columns:
    df['prostate_volume_bin'] = bin_numeric(
        df['prostate_volume'],
        bins=[-np.inf, 30, 50, 80, np.inf],
        labels=['<30 ml', '30-50 ml', '50-80 ml', '>80 ml'],
    )
    schema['prostate_volume_bin'] = ColSpec(
        name='prostate_volume_bin', kind='ordinal',
        ordered_levels=['<30 ml', '30-50 ml', '50-80 ml', '>80 ml'],
    )

# ── 4. Preop PSA (ng/ml) — standard NCCN risk strata ─────────────────────
if 'preop_PSA' in df.columns:
    df['PSA_bin'] = bin_numeric(
        df['preop_PSA'],
        bins=[-np.inf, 4, 10, 20, np.inf],
        labels=['<4', '4-10', '10-20', '>20'],
    )
    schema['PSA_bin'] = ColSpec(
        name='PSA_bin', kind='ordinal',
        ordered_levels=['<4', '4-10', '10-20', '>20'],
    )

# ── 5. Age (years) ───────────────────────────────────────────────────────
if 'age' in df.columns:
    df['age_bin'] = bin_numeric(
        df['age'],
        bins=[-np.inf, 60, 70, 80, np.inf],
        labels=['<60', '60-69', '70-79', '≥80'],
    )
    schema['age_bin'] = ColSpec(
        name='age_bin', kind='ordinal',
        ordered_levels=['<60', '60-69', '70-79', '≥80'],
    )

# ── 6. PIRADS max — needs max_PIRADS column first ────────────────────────
# `mark_structural_missing` already created `n_lesions_MRI_max` (= max PIRADS).
# Rename for clarity if not done already.
if 'n_lesions_MRI_max' in df.columns and 'max_PIRADS' not in df.columns:
    df = df.rename(columns={'n_lesions_MRI_max': 'max_PIRADS'})
    if 'n_lesions_MRI_max' in schema:
        spec = schema.pop('n_lesions_MRI_max')
        spec.name = 'max_PIRADS'
        schema['max_PIRADS'] = spec

if 'max_PIRADS' in df.columns:
    df['max_PIRADS_bin'] = bin_numeric(
        df['max_PIRADS'],
        bins=[-np.inf, 3, 4, np.inf],
        labels=['≤3', '4', '5'],
    )
    schema['max_PIRADS_bin'] = ColSpec(
        name='max_PIRADS_bin', kind='ordinal',
        ordered_levels=['≤3', '4', '5'],
    )


df.filter(regex='_bin$|^psa_cat$').head()

,biopsy_to_RPE_month_bin,PSAD_bin,prostate_volume_bin,PSA_bin,age_bin,max_PIRADS_bin
0,6-9 mo,≥0.15,30-50 ml,10-20,<60,5
2,3-6 mo,<0.15,30-50 ml,4-10,<60,4
3,3-6 mo,<0.15,50-80 ml,4-10,70-79,5
4,6-9 mo,<0.15,50-80 ml,4-10,60-69,5
5,3-6 mo,<0.15,30-50 ml,4-10,60-69,5


## 8. DDA — second pass (with derived columns)

Re-run DDA so the new columns get their own plots and stats.


In [16]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)
#dda_tables['categorical']

print(df['RPE_grade'].value_counts(dropna=False))
print(df['RPE_grade'].nunique())

RPE_grade
2    683
3    179
1    174
5     58
4     41
Name: count, dtype: int64
5


## 9. Configure targets and predictors

This is the only place outcome variables and candidate predictors are declared.


In [17]:
TARGETS = ['upgrade', 'upstage', 'downgrade', 'resection_lines_pos']

# Optional whitelist — leave None to use every testable column except the targets.
PREDICTORS = ['biopsy_type',
    'age_bin', 'PSA_bin', 'prostate_volume_bin', 'PSAD_bin', 'biopsy_to_RPE_month_bin', 'max_PIRADS', 'n_lesions_MRI',
    'risk_group',
    'biopsy_gleason_grade',
]
PREDICTORS = [c for c in PREDICTORS if c in df.columns]  # drop any missing names

# Which value of each target counts as "the event" (positive class).
POSITIVE_CLASS = {t: True for t in TARGETS}


## 10. EDA — univariate screening

For each (target × predictor) pair the engine picks the right test:

| predictor kind | test                     | effect size           |
|----------------|--------------------------|-----------------------|
| continuous     | Mann–Whitney U           | rank-biserial r       |
| ordinal        | Spearman ρ               | ρ                     |
| nominal        | χ² (Fisher if any E<5)   | Cramér's V            |
| binary         | Fisher exact (2×2)       | odds ratio + V        |
| datetime       | MWU on days-since-min    | rank-biserial r       |

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [18]:
assoc = screen_associations(
    df, schema,
    targets=TARGETS,
    predictors=PREDICTORS,
    positive_class=POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

assoc[assoc.fdr_significant]


,target,predictor,kind,test,stat,p,p_fdr,fdr_significant,effect,effect_label,direction,n_used,positive_class
0,downgrade,biopsy_gleason_grade,ordinal,spearman,0.433005,4.503310e-53,4.503310e-52,True,0.433005,spearman_rho,1.0,1135,True
1,downgrade,risk_group,ordinal,spearman,0.215472,2.175395e-13,1.087698e-12,True,0.215472,spearman_rho,1.0,1135,True
10,resection_lines_pos,PSA_bin,ordinal,spearman,0.108901,2.371583e-04,2.371583e-03,True,0.108901,spearman_rho,1.0,1135,True
20,upgrade,biopsy_gleason_grade,ordinal,spearman,-0.463459,1.642491e-61,1.642491e-60,True,-0.463459,spearman_rho,-1.0,1135,True
21,upgrade,risk_group,ordinal,spearman,-0.140752,1.932162e-06,9.660808e-06,True,-0.140752,spearman_rho,-1.0,1135,True
22,upgrade,PSA_bin,ordinal,spearman,0.099249,8.131785e-04,2.070463e-03,True,0.099249,spearman_rho,1.0,1135,True
23,upgrade,biopsy_to_RPE_month_bin,ordinal,spearman,0.099100,8.281853e-04,2.070463e-03,True,0.099100,spearman_rho,1.0,1135,True
24,upgrade,biopsy_type,nominal,chi2,6.038708,1.399552e-02,2.799104e-02,True,0.072941,cramers_v,NaN,1135,True
30,upstage,biopsy_type,nominal,chi2,17.689228,2.600954e-05,2.600954e-04,True,0.124841,cramers_v,NaN,1135,True
31,upstage,PSAD_bin,ordinal,spearman,0.098420,8.996964e-04,4.498482e-03,True,0.098420,spearman_rho,1.0,1135,True


In [19]:
# Full table
assoc


,target,predictor,kind,test,stat,p,p_fdr,fdr_significant,effect,effect_label,direction,n_used,positive_class
0,downgrade,biopsy_gleason_grade,ordinal,spearman,0.433005,4.503310e-53,4.503310e-52,True,0.433005,spearman_rho,1.0,1135,True
1,downgrade,risk_group,ordinal,spearman,0.215472,2.175395e-13,1.087698e-12,True,0.215472,spearman_rho,1.0,1135,True
2,downgrade,biopsy_to_RPE_month_bin,ordinal,spearman,-0.042927,1.483770e-01,4.945899e-01,False,-0.042927,spearman_rho,-1.0,1135,True
3,downgrade,biopsy_type,nominal,chi2,0.483736,4.867350e-01,5.943701e-01,False,0.020645,cramers_v,NaN,1135,True
4,downgrade,PSA_bin,ordinal,spearman,0.028665,3.346255e-01,5.943701e-01,False,0.028665,spearman_rho,1.0,1135,True
5,downgrade,prostate_volume_bin,ordinal,spearman,0.018437,5.349331e-01,5.943701e-01,False,0.018437,spearman_rho,1.0,1135,True
6,downgrade,PSAD_bin,ordinal,spearman,0.023136,4.361633e-01,5.943701e-01,False,0.023136,spearman_rho,1.0,1135,True
7,downgrade,max_PIRADS,ordinal,spearman,-0.034462,2.460197e-01,5.943701e-01,False,-0.034462,spearman_rho,-1.0,1135,True
8,downgrade,n_lesions_MRI,ordinal,spearman,-0.021412,4.711187e-01,5.943701e-01,False,-0.021412,spearman_rho,-1.0,1135,True
9,downgrade,age_bin,ordinal,spearman,-0.003576,9.042174e-01,9.042174e-01,False,-0.003576,spearman_rho,-1.0,1135,True


## 11. Multiple imputation (MICE)

We generate **m=10** imputed datasets via sklearn's IterativeImputer
(RandomForest estimator, separate random seed per imputation).
The pooled inferential stage applies Rubin's rules over these 10 fits.

For a quick screening run set `m=3`. For publication use `m≥10`.


In [20]:
M = 10  # number of imputations; reduce to 3 for fast iteration

imputed_frames = mice_impute(df, schema, m=M, max_iter=10,
                             random_state=42, output_root=OUTPUT_ROOT)
print(f"Generated {len(imputed_frames)} imputed frames")
print("NaN count in first imputed frame:", imputed_frames[0].isna().sum().sum())


Generated 10 imputed frames
NaN count in first imputed frame: 1947


## 12. Multivariable logistic regression (Rubin-pooled)

For each target:

1. Build design matrix (continuous z-scored, ordinal kept as codes, nominal one-hot).
2. Iteratively drop predictors with **VIF > 5** to handle collinearity.
3. Fit logistic regression on each of the m imputed frames.
4. Pool coefficients with **Rubin's rules** (Barnard–Rubin df).
5. Report adjusted OR with 95% CI and pooled p-value.
6. Save a forest plot SVG per target.


In [21]:
inf_results = run_inferential(
    imputed_frames, schema,
    targets=TARGETS,
    predictors=PREDICTORS,
    positive_class=POSITIVE_CLASS,
    vif_threshold=5.0,
    output_root=OUTPUT_ROOT,
)
inf_results


,target,predictor_col,or,or_ci_lo,or_ci_hi,coef,se,p,df,n_models
0,upgrade,biopsy_type_transrektāla,1.241878,0.919774,1.676781,0.216624,0.153192,0.157342,inf,10
1,upgrade,age_bin,1.230290,0.993623,1.523329,0.207250,0.109006,0.057266,inf,10
2,upgrade,PSA_bin,1.714010,1.332380,2.204949,0.538836,0.128507,0.000028,1.081449e+61,10
3,upgrade,prostate_volume_bin,1.119548,0.917693,1.365803,0.112925,0.101439,0.265609,1.719831e+64,10
4,upgrade,PSAD_bin,1.165732,0.787896,1.724760,0.153349,0.199870,0.442938,inf,10
5,upgrade,biopsy_to_RPE_month_bin,1.078213,0.946264,1.228560,0.075305,0.066602,0.258197,inf,10
6,upgrade,max_PIRADS,1.305504,1.067984,1.595848,0.266589,0.102459,0.009271,inf,10
7,upgrade,n_lesions_MRI,0.825415,0.625345,1.089495,-0.191869,0.141627,0.175497,inf,10
8,upgrade,risk_group,0.861275,0.682033,1.087624,-0.149341,0.119051,0.209687,2.039284e+63,10
9,upgrade,biopsy_gleason_grade,0.260566,0.206473,0.328830,-1.344900,0.118720,0.000000,4.923558e+59,10


In [22]:
# Significant adjusted predictors per target
inf_results[(inf_results['p'] < 0.05) & inf_results['or'].notna()]


,target,predictor_col,or,or_ci_lo,or_ci_hi,coef,se,p,df,n_models
2,upgrade,PSA_bin,1.714010,1.332380,2.204949,0.538836,0.128507,0.000028,1.081449e+61,10
6,upgrade,max_PIRADS,1.305504,1.067984,1.595848,0.266589,0.102459,0.009271,inf,10
9,upgrade,biopsy_gleason_grade,0.260566,0.206473,0.328830,-1.344900,0.118720,0.000000,4.923558e+59,10
10,upstage,biopsy_type_transrektāla,0.645180,0.499948,0.832600,-0.438227,0.130117,0.000757,inf,10
14,upstage,PSAD_bin,1.546756,1.095114,2.184663,0.436160,0.176177,0.013298,inf,10
17,upstage,n_lesions_MRI,0.741248,0.582043,0.943999,-0.299420,0.123365,0.015219,1.469543e+62,10
18,upstage,risk_group,0.639752,0.512199,0.799070,-0.446674,0.113455,0.000082,1.051271e+62,10
19,upstage,biopsy_gleason_grade,1.194609,1.033039,1.381448,0.177819,0.074141,0.016467,3.067416e+62,10
26,downgrade,max_PIRADS,0.631321,0.470171,0.847704,-0.459941,0.150369,0.002223,3.243786e+62,10
29,downgrade,biopsy_gleason_grade,3.301869,2.600436,4.192504,1.194489,0.121844,0.000000,inf,10


## 13. **REPORT**


In [23]:
from report import build_report, ReportConfig, write_html
from pathlib import Path
cfg = ReportConfig(output_root=Path("output"), title="...", author="Andy",
                   targets=tuple(TARGETS))
write_html(build_report(cfg), Path("output/report/report.html"))

PosixPath('/Users/andriszaguzovs/TheLibraryOfCode/RPE_petijums/dev/output/report/report.html')

## 13. Outputs

Everything is saved on disk:

```
output/
├── dda/{figures,tables}/
├── missingness/{figures,tables}/
├── eda/{figures,tables}/
└── inferential/{figures,tables}/
```

Each plot is an individual `.svg`; each result table an individual `.csv`.


In [24]:
from pathlib import Path
for p in sorted(Path(OUTPUT_ROOT).rglob('*')):
    if p.is_file():
        print(p)


output/Icon
output/dda/Icon
output/dda/figures/Icon
output/dda/figures/PSAD_bin_015__bar.svg
output/dda/figures/PSAD_bin__bar.svg
output/dda/figures/PSA_bin__bar.svg
output/dda/figures/PSA_density__box.svg
output/dda/figures/PSA_density__hist.svg
output/dda/figures/RPE_TNM__bar.svg
output/dda/figures/RPE_grade__bar.svg
output/dda/figures/age__box.svg
output/dda/figures/age__hist.svg
output/dda/figures/age_bin__bar.svg
output/dda/figures/biopsy_gleason_grade__bar.svg
output/dda/figures/biopsy_to_RPE_days__box.svg
output/dda/figures/biopsy_to_RPE_days__hist.svg
output/dda/figures/biopsy_to_RPE_month_bin__bar.svg
output/dda/figures/biopsy_type__bar.svg
output/dda/figures/downgrade__bar.svg
output/dda/figures/lesion_1_MRI_PIRADS__bar.svg
output/dda/figures/lesion_2_MRI_PIRADS__bar.svg
output/dda/figures/lesion_3_MRI_PIRADS__bar.svg
output/dda/figures/max_PIRADS__bar.svg
output/dda/figures/max_PIRADS_bin__bar.svg
output/dda/figures/n_lesions_MRI__bar.svg
output/dda/figures/preop_PSA__box.sv

## 14. NOTES — Why each statistical choice

Concise but detailed rationale for every formula used in this pipeline.
For each: **what it does**, **why chosen**, **what was rejected**.

---

### Schema inference (hybrid auto + override)

- **What.** Heuristic classification of each column into `continuous / count / ordinal / nominal / binary / datetime / id / text / skip` based on dtype, cardinality, value patterns.
- **Why.** Test selection downstream is kind-driven — a wrong kind silently picks the wrong test (e.g. treating Gleason 1–5 as `continuous` instead of `ordinal` swaps Spearman for MWU and loses interpretability of "per-grade increase").
- **Alternatives rejected.**
  - *Full auto-only*: brittle on clinical data where 0/1-coded ordinals look numeric.
  - *Manual ColSpec per column*: correct but tedious; you'd re-type 30+ specs per study.

---

### Duplicate auditing on normalized string keys

- **What.** Lowercase + strip + empty→NA on ID columns, then flag rows whose full key tuple is non-null and repeated.
- **Why.** Clinical IDs (`pk`, `year`) frequently have invisible whitespace or case drift across data-entry sessions. Naive `duplicated()` misses these.
- **Alternatives rejected.**
  - *Exact match*: under-detects.
  - *Fuzzy match (Levenshtein)*: over-detects, would falsely merge genuinely different patients.

---

### Mann–Whitney U for continuous/count vs binary outcome

- **What.** Non-parametric rank-sum test. H₀: P(X₁ > X₂) = ½. Two-sided.
- **Effect size.** Rank-biserial **r = |Z|/√N**, where Z is the large-sample normal approximation of U. Bounded 0–1, interpretable like Cohen's r (0.1 small, 0.3 medium, 0.5 large).
- **Why.**
  - Clinical continuous variables (PSA, age, days-to-surgery) are **almost never normal** — PSA in particular is heavily right-skewed.
  - MWU has ~95% efficiency vs t-test under normality and is far more robust under non-normality.
  - One test for the whole pipeline = no test-switching artifacts.
- **Alternatives rejected.**
  - *Welch's t-test always*: violates assumption on skewed data; inflates type-I error on small skewed samples.
  - *Auto Shapiro-Wilk switch (t if normal, MWU else)*: the normality test itself adds noise and its decision is sample-size dependent (always rejects normal at large N, never at small N) — produces worse calibration than just using MWU.
  - *Welch's t on log-transformed data*: works for PSA specifically but not generalizable to all continuous predictors in the pipeline.
- **Sensitivity.** When publishing, re-run Welch's t on log(PSA) as a sensitivity analysis — if direction and significance agree with MWU, you're robust.

---

### Spearman ρ for ordinal vs binary outcome

- **What.** Pearson correlation on the ranks of category codes vs the 0/1-encoded outcome.
- **Why.**
  - Preserves the **ordering** of ordinal predictors (Gleason 1<2<3<4<5, PIRADS 1<2<3<4<5, risk_group low<mid<high). χ² throws this away — it would only tell you "the distribution differs across levels", not "higher Gleason → more upgrades".
  - Yields a signed, scale-free effect size (ρ) that's directly publishable.
- **Alternatives rejected.**
  - *χ² on the ordinal × binary table*: ignores ordering, weaker power, no direction.
  - *Cochran-Armitage trend test*: equivalent to a linear-trend variant of χ² and gives p only — Spearman gives p **plus** a comparable ρ across all ordinal predictors.
  - *Kendall's τ*: similar info but slower on large N and no power advantage here.

---

### χ² (or Fisher exact) for nominal vs binary

- **What.** χ² of independence on the contingency table, **without Yates correction** (modern recommendation — Yates is overconservative). Switches to **Fisher exact** if the 2×2 table has any expected cell count < 5.
- **Why Fisher when expected<5.** χ²'s asymptotic distribution breaks down with small expected counts; Fisher's exact test conditions on the marginals and computes the exact hypergeometric p — correct at any sample size.
- **Effect size: Cramér's V** = √(χ²/(N·(min(r,c)−1))). Bounded 0–1, comparable across table shapes. For 2×2 tables we **also** report the odds ratio because clinicians read OR natively.
- **Alternatives rejected.**
  - *Yates-corrected χ²*: too conservative for modern computing — Fisher is exact and almost as fast.
  - *G-test (likelihood ratio)*: theoretically nicer for nested models but identical conclusions in 2-way tables; less familiar to clinical readers.
  - *Permutation χ²*: same answer as Fisher for 2×2, more expensive.

---

### Benjamini–Hochberg FDR correction, per target

- **What.** Sort p-values ascending; for rank i out of m, compute q_i = p_(i)·m/i; enforce monotonicity from the right; significance at q < α controls expected proportion of false discoveries at α.
- **Why per-target (not pooled across all targets).** Each outcome (upgrade, upstage, downgrade) is a **separate family** of hypotheses with its own scientific interpretation. Pooling them inflates the family size and over-corrects. This matches how clinical journals report multi-outcome studies.
- **Alternatives rejected.**
  - *Bonferroni*: controls family-wise error rate — far too conservative for a screening stage with 10+ predictors. Misses real signal.
  - *Holm-Bonferroni*: still FWER, marginally less conservative than Bonferroni but still much stricter than BH.
  - *Storey q-value*: estimates the null proportion adaptively; great when you have hundreds of tests but unstable at small m (you'll have <20 tests per target).
  - *No correction*: indefensible with ≥3 predictors per target — false discovery rate would be ~30%+.
- **Verified.** Output matches `statsmodels.stats.multitest.multipletests(method='fdr_bh')` exactly.

---

### MICE (Multiple Imputation by Chained Equations), m=10

- **What.** For each missing value: fit a regression of that column on all others using observed data, predict missing values, iterate until convergence. Repeat with m different random seeds to produce m plausible completed datasets.
- **Why multiple (not single).** Single imputation pretends the imputed values are known, so it **understates standard errors**. With m=10 imputations and Rubin pooling, the SEs honestly include imputation uncertainty.
- **Estimator: RandomForestRegressor.** Captures non-linear relationships (PSA × age × Gleason interactions) without you specifying them. Tolerates mixed numeric/categorical inputs.
- **Why m=10.** Rubin showed efficiency = (1 + fmi/m)^(-1) where fmi is fraction of missing info. At fmi ≈ 0.3 (typical clinical data), m=10 gives ~97% efficiency. m=5 is acceptable, m=20 is overkill.
- **Alternatives rejected.**
  - *Mean/median imputation*: distorts variance and any correlation involving the imputed column. Catastrophic for inferential SEs.
  - *Complete-case analysis*: throws away rows with any missingness — typically 20–50% data loss in clinical cohorts; introduces selection bias if missingness is MAR (which it usually is).
  - *Hot-deck imputation*: works for nominal-only data; weaker for mixed types.
  - *Bayesian model-based imputation (`mice` R package, Stan)*: gold standard but heavy infrastructure; sklearn's `IterativeImputer` is close enough for clinical screening.
- **Limitation.** Assumes data is **Missing At Random** (MAR) — missingness depends only on observed variables. For **MNAR** patterns (e.g. "PSA was missing because risk was low"), add explicit `<col>_missing` flags in section 6a so the model can use the missingness indicator itself as a predictor.

---

### Missingness flags

- **What.** Binary indicator columns `<col>_missing` added before imputation.
- **Why.** In clinical data, *that a value was missing* is often informative (e.g. PSA not measured because clinician judged it unnecessary). Including the flag in the regression lets the model separate "the value's effect" from "the act of measuring's effect".
- **Alternatives rejected.**
  - *Imputing without flags*: hides the MNAR mechanism.
  - *Dropping columns with high missingness*: throws away signal; missingness % is not a reliable filter for clinical utility.

---

### Variance Inflation Factor (VIF) pruning, threshold = 5

- **What.** For each predictor x_j, VIF = 1/(1 − R²_j), where R²_j is from regressing x_j on all other predictors. Iteratively drop the column with the highest VIF until all ≤ 5.
- **Why.** Logistic regression with collinear predictors produces enormous standard errors and unstable coefficients ("model can't tell whether PSA or PSA-density is doing the work"). VIF > 5 ⇔ R²_j > 0.80 ⇔ severe multicollinearity.
- **Why threshold = 5** (not 10). VIF=10 is the classical statistics teaching threshold but for clinical regression with modest N (<500), 5 is the modern recommendation (Vatcheva 2016, O'Brien 2007).
- **Alternatives rejected.**
  - *Pairwise Pearson correlation > 0.8*: catches only 2-variable collinearity; misses 3-way (e.g. a = b + c).
  - *Lasso regularization*: would drop collinear features automatically but **biases coefficients** toward zero — bad for inference (you want unbiased OR estimates). Lasso is for prediction, not inference.
  - *Ridge / Elastic Net*: same problem — shrinks coefficients, distorts ORs.
  - *PCA / partial-least-squares*: components are uninterpretable clinically.

---

### Multivariable logistic regression

- **What.** Per target, one binary logistic model with all surviving predictors. Continuous z-scored (so OR is per-SD increase), ordinals kept as numeric codes, nominals one-hot with drop_first.
- **Why.** Univariate screening (section 10) ignores confounding — Gleason can show up "significant" purely because it correlates with PSA. Multivariable estimates the **adjusted** effect of each predictor holding the others constant.
- **Why this design encoding.**
  - *z-score continuous*: ORs comparable across predictors; one "unit" = one SD.
  - *Ordinal as numeric code*: assumes linear log-odds across levels (parsimonious; standard for Gleason/PIRADS in urology papers). The alternative is one-hot with drop_first, which uses more degrees of freedom and is only worth it if the trend is clearly non-monotonic — check the EDA plots first.
  - *Nominal one-hot drop_first*: avoids the dummy variable trap (perfect collinearity with intercept).
- **Alternatives rejected.**
  - *Univariate-only pipeline*: misleading because of confounding.
  - *Stepwise selection (forward/backward)*: notorious for unstable selection, inflated significance, and irreproducibility. Modern guidance (Harrell, Steyerberg) is: don't.
  - *Random forest / XGBoost*: better predictive accuracy but no clinical OR with CI to report.
  - *Penalized regression (Firth, Lasso, Ridge)*: useful with extreme separation or n<<p but biases the OR estimates — defeats the inferential purpose.
  - *Bayesian logistic with weakly informative priors*: cleaner for tiny samples and would give credible intervals — but you'd need to defend prior choice in the manuscript.

---

### Rubin's rules with Barnard–Rubin degrees of freedom

- **What.** Across the m=10 imputed-frame fits, for each coefficient:
  - θ̄ = mean of the m point estimates
  - within-imp variance Ū = mean of the m squared SEs
  - between-imp variance B = sample variance of the m estimates
  - total variance T = Ū + (1 + 1/m)·B
  - pooled SE = √T
  - degrees of freedom (Barnard–Rubin):
    df = (m−1)·(1 + Ū/((1+1/m)·B))²
  - p-value from t-distribution with that df; 95% CI = θ̄ ± t_{0.975, df}·SE
- **Why.** Rubin's rules are the **only** statistically valid way to combine results across multiple imputations. The total variance T splits into "within" (each model's uncertainty) and "between" (uncertainty due to missing data) — they're not interchangeable.
- **Why Barnard–Rubin df (not the original Rubin 1987 df).** Original Rubin df → ∞ when between-variance is small, which is wrong when m is small. Barnard–Rubin (1999) is a small-sample correction that's now the standard (R `mice` uses it, SAS PROC MIANALYZE uses it).
- **Alternatives rejected.**
  - *Picking the "best" imputation*: defeats the purpose of multiple imputation entirely.
  - *Average the imputed datasets first, then fit once*: produces correct point estimates but **wrong SEs** (the between-variance is invisible).
  - *Use the within-variance only*: ignores imputation uncertainty — false confidence.
- **Verified.** With zero between-variance, our pooler returns SE equal to the single-fit SE; with non-zero between, it correctly inflates SE and produces a finite small-sample df (e.g. m=5, modest B → df ≈ 22).

---

### Why log-scale x-axis on forest plots

- ORs are multiplicative (OR=2 and OR=0.5 are equal-and-opposite effects). On a linear axis they look asymmetric; on log scale they're symmetric around OR=1, which is the correct visual.

---

### What this pipeline deliberately does NOT do

- **No machine-learning prediction** (no train/test split, no AUC, no calibration). This is an **association/inference** pipeline, not a prediction pipeline. If you later want a predictive model (e.g. nomogram for upgrade risk), that's a separate workflow with cross-validation, calibration plots, decision-curve analysis.
- **No causal inference** (no DAGs, no IPTW, no instrumental variables). All effects here are **statistical associations** adjusted for the included covariates — they are *not* causal effects. Manuscript wording must say "associated with", never "causes".
- **No survival/time-to-event analysis.** Targets here are binary (upgrade yes/no). If you later care about *time to biochemical recurrence*, you'd need Cox regression — a separate module.

---

### Sanity-check checklist before submitting results

1. Print `schema_summary(schema)` — every ordinal has correct `ordered_levels`?
2. After MICE, `imputed_frames[0].isna().sum().sum()` == 0 for predictor columns?
3. `inf_results['n_models']` ≈ m for all predictors (means the model converged on every imputation)?
4. Forest plot ORs and EDA univariate effects agree in **direction** (sign)? If they flip, you have confounding worth discussing.
5. For each FDR-significant univariate result, check the corresponding plot in `output/eda/figures/` — is the pattern visually credible or driven by 2–3 outliers?
